# Lab 2 — Evaluate Retrieval and Grounded Answers

**Required · 50 minutes**

Build a small quality baseline for an internal-operations assistant using pre-computed queries, retrieved context, and responses. The dataset is intentionally deterministic; one weak row makes the difference between retrieval quality, groundedness, and answer relevance visible.

**Artifact:** a Foundry evaluation report plus a short interpretation of the weakest row.

## Evaluation starts after retrieval

Foundry IQ or Azure AI Search can retrieve enterprise knowledge. These evaluators do not retrieve or index documents:

- **Retrieval** — is the supplied context useful for the query?
- **Groundedness** — is the response supported by that context?
- **Relevance** — does the response address the query?

A response can be relevant but ungrounded, or grounded while its retrieved context is irrelevant.

In [ ]:
import os
import re
import time
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

def load_repo_env() -> Path | None:
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        if (folder / '.env').exists():
            load_dotenv(folder / '.env')
            return folder / '.env'
    return None

load_repo_env()
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
team_id = os.getenv('WORKSHOP_TEAM_ID', '').strip()
participant_id = os.getenv('WORKSHOP_PARTICIPANT_ID', '').strip()
configured_namespace = os.getenv('WORKSHOP_RESOURCE_NAMESPACE', '').strip()
raw_namespace = configured_namespace or team_id or participant_id
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not endpoint or not model_deployment or not resource_namespace:
    raise ValueError('Missing Foundry endpoint, model deployment, or workshop namespace')

sdk_version = tuple(int(p) for p in version('azure-ai-projects').split('.')[:2])
if sdk_version < (2, 2):
    raise RuntimeError('This lab targets azure-ai-projects>=2.2.0')
print({'namespace': resource_namespace, 'model': model_deployment, 'azure-ai-projects': version('azure-ai-projects')})

## 1. Deterministic workshop dataset

All procedure text and incident identifiers below are synthetic. In a real test set, store dataset versions and preserve representative failure cases.

In [ ]:
quality_rows = [
    {
        'case_id': 'Q-01',
        'query': 'What must be verified before dispatch for synthetic incident SIM-1042?',
        'context': 'Procedure P-17: verify isolation and absence of voltage, then obtain switching-authority confirmation before dispatch.',
        'response': 'Before dispatch, verify isolation and absence of voltage and obtain switching-authority confirmation (P-17).',
        'ground_truth': 'Verify isolation, absence of voltage, and switching-authority confirmation under P-17.',
        'expected_quality': 'pass',
    },
    {
        'case_id': 'Q-02',
        'query': 'Which synthetic procedure applies to a damaged service cabinet?',
        'context': 'Procedure P-22 covers damaged service cabinets. Procedure P-17 covers switching preparation.',
        'response': 'Procedure P-22 applies to a damaged service cabinet.',
        'ground_truth': 'P-22.',
        'expected_quality': 'pass',
    },
    {
        'case_id': 'Q-03',
        'query': 'Which role confirms readiness before dispatch under P-17?',
        'context': 'P-17 states that the switching authority is the role responsible for confirming readiness before dispatch. The policy does not name an individual.',
        'response': 'The site manager, Jordan Lee, confirms readiness.',
        'ground_truth': 'The switching authority confirms readiness; no person is named.',
        'expected_quality': 'fail',
    },
    {
        'case_id': 'Q-04',
        'query': 'According to the supplied policy, can the assistant confirm that dispatch has already happened?',
        'context': 'Policy: the assistant provides guidance only, so it cannot execute an operational action or confirm that dispatch has already happened.',
        'response': 'No. It can provide guidance but cannot execute or confirm that dispatch occurred.',
        'ground_truth': 'No; the assistant cannot execute or confirm operational actions.',
        'expected_quality': 'pass',
    },
]

assert len({row['case_id'] for row in quality_rows}) == len(quality_rows)
assert [row['case_id'] for row in quality_rows if row['expected_quality'] == 'fail'] == ['Q-03']
assert all(row['query'] and row['context'] and row['response'] for row in quality_rows)
print('PASS — deterministic dataset contract is valid.')

## 2. Configure the cloud evaluation

The current cloud evaluation SDK uses a custom item schema and `{{item.field}}` mappings. Inline `file_content` keeps this timed lab independent of shared dataset resources.

In [ ]:
from azure.identity import InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

credential = InteractiveBrowserCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'case_id': {'type': 'string'},
            'query': {'type': 'string'},
            'context': {'type': 'string'},
            'response': {'type': 'string'},
            'ground_truth': {'type': 'string'},
            'expected_quality': {'type': 'string'},
        },
        'required': ['case_id', 'query', 'context', 'response', 'ground_truth'],
    },
)

def judge(name: str, evaluator_name: str, mapping: dict):
    return TestingCriterionAzureAIEvaluator(
        type='azure_ai_evaluator',
        name=name,
        evaluator_name=evaluator_name,
        initialization_parameters={'deployment_name': model_deployment},
        data_mapping=mapping,
    )

testing_criteria = [
    judge('retrieval', 'builtin.retrieval', {'query': '{{item.query}}', 'context': '{{item.context}}'}),
    judge('groundedness', 'builtin.groundedness', {'query': '{{item.query}}', 'context': '{{item.context}}', 'response': '{{item.response}}'}),
    judge('relevance', 'builtin.relevance', {'query': '{{item.query}}', 'response': '{{item.response}}'}),
]
print('Configured:', [criterion['name'] for criterion in testing_criteria])

## 3. Create and run the baseline

In [ ]:
eval_name = f'd2-quality-{resource_namespace}'
run_name = f'd2-quality-baseline-{resource_namespace}'
eval_object = openai_client.evals.create(
    name=eval_name,
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=run_name,
    metadata={'namespace': resource_namespace, 'team_id': team_id, 'dataset': 'synthetic-ops-v1'},
    data_source={
        'type': 'jsonl',
        'source': {'type': 'file_content', 'content': [{'item': row} for row in quality_rows]},
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('Evaluation exceeded 20 minutes; cancel it or check model capacity')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)

output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
if eval_run.status != 'completed':
    run_error = getattr(eval_run, 'error', None)
    if hasattr(run_error, 'model_dump'):
        run_error = run_error.model_dump(mode='json')
    raise RuntimeError(f'Evaluation infrastructure failure: {{"status": {eval_run.status!r}, "eval_id": {eval_object.id!r}, "run_id": {eval_run.id!r}, "server_error": {run_error!r}, "output_items": {len(output_items)}, "report_url": {getattr(eval_run, "report_url", None)!r}}}')
output_deadline = time.monotonic() + 2 * 60
while len(output_items) < len(quality_rows):
    if time.monotonic() > output_deadline:
        raise TimeoutError(f'Evaluation completed but exposed only {len(output_items)}/{len(quality_rows)} output items')
    time.sleep(2)
    output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
assert len(output_items) == len(quality_rows)
failed_results = [
    result
    for item in output_items
    for result in item.model_dump(mode='json')['results']
    if result.get('error') or result.get('status') in ('failed', 'error', 'canceled')
]
assert not failed_results, failed_results
print({'status': eval_run.status, 'output_items': len(output_items), 'report_url': getattr(eval_run, 'report_url', None)})

## 4. Inspect and interpret

LLM-assisted scores may vary. Review each failed example and its reason instead of treating an aggregate score as a release decision.

In [ ]:
def primitive(value):
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if hasattr(value, 'to_dict'):
        return value.to_dict()
    return value

for item in output_items:
    data = primitive(item)
    print('\nCASE:', data.get('item_id') or data.get('id') or 'result')
    print(data)

assert output_items, 'A completed run should contain scored output items'
print('\nPASS — the cloud evaluation completed and returned scored items.')

## Participant challenge

Fix only `Q-03` so it names the role supported by context and explicitly abstains from naming a person. Create a second run under the **same evaluation**, using the run name `d2-quality-fixed-<namespace>`. Compare the two report URLs.

Your release note must answer:

1. Which metric exposed the unsupported name most clearly?
2. Did retrieval need to change, or only generation?
3. Would you ship based on four rows? Why not?

In [ ]:
# TODO: copy quality_rows, repair Q-03, and create a second run.
# fixed_rows = ...
# fixed_run = openai_client.evals.runs.create(...)


## Optional extension

Add Response Completeness (preview) with `ground_truth` and `response`, or replace the pre-computed context with context captured from a versioned Foundry IQ knowledge base. Keep the retrieval system version in run metadata.

## Cleanup (opt-in and namespace-safe)

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
if allow_cleanup:
    if not eval_name.endswith(f'-{resource_namespace}'):
        raise RuntimeError(f'Refusing to delete non-owned evaluation: {eval_name}')
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation and its runs.')
else:
    print('Cleanup disabled so reports remain available for comparison.')